# Deep Learning Toolkit on the Disease–Symptom Dataset

`Training.csv` has 4,920 rows and `Testing.csv` has 42 rows. Each row holds 132 binary symptom flags and a `prognosis` label with 41 diseases.

1. PyTorch basics: tensors, autograd, training loop
2. CNNs
3. RNNs / LSTMs / GRUs
4. Regularization: dropout, batch norm, early stopping
5. Transfer learning with a pretrained CNN
6. Hugging Face transformers
7. LLM APIs: OpenAI, Anthropic, Hugging Face, Ollama

Educational use only. Nothing here is medical advice.

In [ ]:
%pip install -q torch torchvision transformers scikit-learn pandas matplotlib requests openai anthropic huggingface_hub

## 0. Setup and data

In [ ]:
import os, copy, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(device)

In [ ]:
TRAIN_PATH = "Training.csv"
TEST_PATH = "Testing.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

train_df = train_df.loc[:, ~train_df.columns.str.startswith("Unnamed")]
train_df.columns = train_df.columns.str.strip()
test_df.columns = test_df.columns.str.strip()

TARGET = "prognosis"
feature_cols = [c for c in train_df.columns if c != TARGET]
assert feature_cols == [c for c in test_df.columns if c != TARGET]

train_df[TARGET] = train_df[TARGET].str.strip()
test_df[TARGET] = test_df[TARGET].str.strip()

print(train_df.shape, test_df.shape)
print(train_df[TARGET].nunique(), "classes")
print(train_df[TARGET].value_counts().describe())
train_df.head()

In [ ]:
le = LabelEncoder().fit(train_df[TARGET])
n_classes = len(le.classes_)
n_features = len(feature_cols)

X_all = torch.tensor(train_df[feature_cols].values, dtype=torch.float32)
y_all = torch.tensor(le.transform(train_df[TARGET]), dtype=torch.long)
X_test = torch.tensor(test_df[feature_cols].values, dtype=torch.float32)
y_test = torch.tensor(le.transform(test_df[TARGET]), dtype=torch.long)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=SEED
)

print(X_tr.shape, X_val.shape, X_test.shape)
print("duplicate training rows:", train_df.duplicated().sum())
print("avg symptoms per row:", X_all.sum(1).mean().item())

In [ ]:
def make_loader(X, y, batch_size=64, shuffle=False):
    return DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=shuffle)


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = F.cross_entropy(logits, yb)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(yb)
            correct += (logits.argmax(1) == yb).sum().item()
            n += len(yb)
    return total_loss / n, correct / n


def fit(model, train_loader, val_loader, epochs=30, lr=1e-3, weight_decay=0.0,
        patience=None, verbose=True):
    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_loss, best_state, wait = float("inf"), None, 0
    for epoch in range(1, epochs + 1):
        tl, ta = run_epoch(model, train_loader, optimizer)
        vl, va = run_epoch(model, val_loader)
        for k, v in zip(history, (tl, vl, ta, va)):
            history[k].append(v)
        if verbose and (epoch == 1 or epoch % 5 == 0):
            print(f"epoch {epoch:3d} | train loss {tl:.4f} acc {ta:.3f} | val loss {vl:.4f} acc {va:.3f}")
        if patience is not None:
            if vl < best_loss - 1e-4:
                best_loss, wait = vl, 0
                best_state = copy.deepcopy(model.state_dict())
            else:
                wait += 1
                if wait >= patience:
                    if verbose:
                        print(f"early stopping at epoch {epoch}")
                    break
    if best_state is not None:
        model.load_state_dict(best_state)
    return history


@torch.no_grad()
def accuracy(model, X, y):
    model.eval()
    preds = []
    for i in range(0, len(X), 256):
        preds.append(model(X[i:i + 256].to(device)).argmax(1).cpu())
    return (torch.cat(preds) == y).float().mean().item()


def plot_history(histories, key="val_loss", title=None):
    plt.figure(figsize=(6, 4))
    for name, h in histories.items():
        plt.plot(h[key], label=name)
    plt.xlabel("epoch")
    plt.ylabel(key)
    plt.title(title or key)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 1. PyTorch basics: tensors, autograd, training loop

### Tensors

In [ ]:
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.randn(2, 2)

print(a.shape, a.dtype, a.device)
print(a + b)
print(a @ b)
print(a.mean(dim=0), a.sum(dim=1))
print(a.reshape(4), a.T)
print(torch.zeros(2, 3), torch.ones(2, 3), torch.arange(6).reshape(2, 3))

sample = X_all[0]
active = torch.nonzero(sample).squeeze(1).tolist()
print([feature_cols[i] for i in active])
print(X_all[:4].shape, y_all[:4], le.inverse_transform(y_all[:4].numpy()))

### Autograd

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 2 * x + 1
y.backward()
print(x.grad)

w = torch.randn(3, requires_grad=True)
inp = torch.tensor([1.0, 2.0, 3.0])
target = torch.tensor(10.0)
loss = (w @ inp - target) ** 2
loss.backward()
print(w.grad)
print(2 * (w @ inp - target) * inp)

with torch.no_grad():
    z = w * 2
print(z.requires_grad)

### Training loop from scratch: softmax regression with manual gradient updates

In [ ]:
W = torch.zeros(n_features, n_classes, requires_grad=True)
b = torch.zeros(n_classes, requires_grad=True)
lr = 0.5

for epoch in range(1, 51):
    logits = X_tr @ W + b
    loss = F.cross_entropy(logits, y_tr)
    loss.backward()
    with torch.no_grad():
        W -= lr * W.grad
        b -= lr * b.grad
    W.grad.zero_()
    b.grad.zero_()
    if epoch % 10 == 0:
        with torch.no_grad():
            val_acc = ((X_val @ W + b).argmax(1) == y_val).float().mean().item()
        print(f"epoch {epoch:3d} | loss {loss.item():.4f} | val acc {val_acc:.3f}")

### The idiomatic version: `nn.Module`, `DataLoader`, optimizer

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=(128, 64)):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


train_loader = make_loader(X_tr, y_tr, shuffle=True)
val_loader = make_loader(X_val, y_val)

mlp = MLP(n_features, n_classes)
print(mlp)
print("trainable parameters:", count_params(mlp))

mlp_hist = fit(mlp, train_loader, val_loader, epochs=15, lr=1e-3)
print("test accuracy:", accuracy(mlp, X_test, y_test))
plot_history({"mlp": mlp_hist}, "val_loss", "MLP validation loss")

In [ ]:
torch.save(mlp.state_dict(), "mlp.pt")
reloaded = MLP(n_features, n_classes)
reloaded.load_state_dict(torch.load("mlp.pt", map_location="cpu"))
reloaded.to(device)
print(accuracy(reloaded, X_test, y_test))

def predict_disease(model, symptoms, to_input=lambda x: x):
    vec = torch.zeros(1, n_features)
    for s in symptoms:
        vec[0, feature_cols.index(s)] = 1.0
    model.eval()
    with torch.no_grad():
        probs = F.softmax(model(to_input(vec).to(device)), dim=1)[0].cpu()
    top = probs.topk(3)
    return [(le.classes_[i], round(p.item(), 3)) for p, i in zip(top.values, top.indices)]

predict_disease(mlp, ["itching", "skin_rash", "nodal_skin_eruptions"])

## 2. CNNs: convolutions, pooling, classification

The 132 symptom flags are padded to 144 and arranged as a 12×12 single-channel image, so convolutions can look at neighbouring columns. Column adjacency has no real meaning in this dataset, so expect the CNN to work but not to beat the MLP for principled reasons.

In [ ]:
def to_grid(X):
    pad = torch.zeros(len(X), 144 - X.shape[1])
    return torch.cat([X, pad], dim=1).view(-1, 1, 12, 12)


G_tr, G_val, G_test = to_grid(X_tr), to_grid(X_val), to_grid(X_test)
print(G_tr.shape)

fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for ax, i in zip(axes, range(4)):
    ax.imshow(G_tr[i, 0], cmap="gray_r")
    ax.set_title(le.classes_[y_tr[i]][:18], fontsize=8)
    ax.axis("off")
plt.show()

In [ ]:
img = G_tr[:1]
kernel = torch.tensor([[[[1.0, 0.0, -1.0], [2.0, 0.0, -2.0], [1.0, 0.0, -1.0]]]])
print("conv (no padding):", F.conv2d(img, kernel).shape)
print("conv (padding=1):", F.conv2d(img, kernel, padding=1).shape)
print("max pool:", F.max_pool2d(img, 2).shape)
print("avg pool:", F.avg_pool2d(img, 2).shape)

In [ ]:
class SymptomCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 3 * 3, 64),
            nn.ReLU(),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


cnn = SymptomCNN(n_classes)
h = G_tr[:2]
for layer in cnn.features:
    h = layer(h)
    print(f"{layer.__class__.__name__:10s} -> {tuple(h.shape)}")
print("trainable parameters:", count_params(cnn))

In [ ]:
cnn_train_loader = make_loader(G_tr, y_tr, shuffle=True)
cnn_val_loader = make_loader(G_val, y_val)

cnn_hist = fit(cnn, cnn_train_loader, cnn_val_loader, epochs=15, lr=1e-3)
print("test accuracy:", accuracy(cnn, G_test, y_test))
plot_history({"cnn": cnn_hist, "mlp": mlp_hist}, "val_loss", "CNN vs MLP validation loss")

In [ ]:
with torch.no_grad():
    filters = cnn.features[0].weight.cpu()
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for ax, f in zip(axes.ravel(), filters):
    ax.imshow(f[0], cmap="coolwarm")
    ax.axis("off")
plt.suptitle("Learned 3x3 filters")
plt.show()

## 3. RNNs, LSTMs and GRUs: sequence modeling basics

Each patient becomes a variable-length sequence of symptom tokens (index 0 is padding). The order follows column order, so this demonstrates the mechanics of sequence models rather than a natural temporal signal.

In [ ]:
MAX_LEN = int(max(X_all.sum(1).max().item(), X_test.sum(1).max().item()))
VOCAB = n_features + 1


def to_seq(X, max_len=MAX_LEN):
    out = torch.zeros(len(X), max_len, dtype=torch.long)
    for i, row in enumerate(X):
        idx = torch.nonzero(row).squeeze(1) + 1
        out[i, :len(idx)] = idx
    return out


S_tr, S_val, S_test = to_seq(X_tr), to_seq(X_val), to_seq(X_test)
print(S_tr.shape, "max length:", MAX_LEN)
print(S_tr[0])
print([feature_cols[i - 1] for i in S_tr[0] if i > 0])

In [ ]:
from torch.nn.utils.rnn import pack_padded_sequence


class SeqClassifier(nn.Module):
    def __init__(self, cell="lstm", vocab=VOCAB, emb=32, hidden=64, n_classes=n_classes,
                 num_layers=1, bidirectional=False, dropout=0.0):
        super().__init__()
        self.cell = cell
        self.embedding = nn.Embedding(vocab, emb, padding_idx=0)
        rnn_cls = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[cell]
        self.rnn = rnn_cls(
            emb, hidden, num_layers=num_layers, batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        out_dim = hidden * (2 if bidirectional else 1)
        self.head = nn.Linear(out_dim, n_classes)
        self.bidirectional = bidirectional

    def forward(self, x):
        lengths = (x != 0).sum(1).clamp(min=1).cpu()
        emb = self.embedding(x)
        packed = pack_padded_sequence(emb, lengths, batch_first=True, enforce_sorted=False)
        _, state = self.rnn(packed)
        h = state[0] if self.cell == "lstm" else state
        if self.bidirectional:
            last = torch.cat([h[-2], h[-1]], dim=1)
        else:
            last = h[-1]
        return self.head(last)


seq_train_loader = make_loader(S_tr, y_tr, shuffle=True)
seq_val_loader = make_loader(S_val, y_val)

seq_models, seq_hists = {}, {}
for cell in ["rnn", "lstm", "gru"]:
    torch.manual_seed(SEED)
    m = SeqClassifier(cell)
    print(f"--- {cell.upper()} | params {count_params(m)}")
    seq_hists[cell] = fit(m, seq_train_loader, seq_val_loader, epochs=20, lr=2e-3)
    seq_models[cell] = m

plot_history(seq_hists, "val_loss", "Recurrent cells: validation loss")

In [ ]:
pd.DataFrame({
    cell: {
        "params": count_params(m),
        "val_acc": accuracy(m, S_val, y_val),
        "test_acc": accuracy(m, S_test, y_test),
    }
    for cell, m in seq_models.items()
}).T

In [ ]:
bi_gru = SeqClassifier("gru", num_layers=2, bidirectional=True, dropout=0.2)
bi_hist = fit(bi_gru, seq_train_loader, seq_val_loader, epochs=20, lr=2e-3)
print("test accuracy:", accuracy(bi_gru, S_test, y_test))

## 4. Regularization: dropout, batch norm, early stopping

The full dataset is nearly separable, so every model reaches ~100% and regularization shows no effect. To make the effect visible, the models train on only 300 clean rows and are validated and tested on rows with randomly flipped symptoms (simulating noisy self-reporting).

In [ ]:
def flip_bits(X, p, seed):
    g = torch.Generator().manual_seed(seed)
    mask = (torch.rand(X.shape, generator=g) < p).float()
    return (X + mask) % 2


X_small, _, y_small, _ = train_test_split(
    X_tr, y_tr, train_size=300, stratify=y_tr, random_state=SEED
)
X_val_noisy = flip_bits(X_val, 0.03, seed=1)
X_test_noisy = flip_bits(X_test, 0.03, seed=2)

small_loader = make_loader(X_small, y_small, batch_size=32, shuffle=True)
noisy_val_loader = make_loader(X_val_noisy, y_val)
print(X_small.shape, X_val_noisy.shape)

In [ ]:
class RegMLP(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=(256, 128), dropout=0.0, batch_norm=False):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden:
            layers.append(nn.Linear(prev, h))
            if batch_norm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


configs = {
    "baseline": dict(model=dict(), fit=dict()),
    "dropout 0.5": dict(model=dict(dropout=0.5), fit=dict()),
    "batch norm": dict(model=dict(batch_norm=True), fit=dict()),
    "weight decay": dict(model=dict(), fit=dict(weight_decay=1e-3)),
    "dropout + bn + wd + early stop": dict(
        model=dict(dropout=0.3, batch_norm=True), fit=dict(weight_decay=1e-4, patience=8)
    ),
}

reg_models, reg_hists = {}, {}
for name, cfg in configs.items():
    torch.manual_seed(SEED)
    m = RegMLP(n_features, n_classes, **cfg["model"])
    print(f"--- {name}")
    reg_hists[name] = fit(m, small_loader, noisy_val_loader, epochs=60, lr=1e-3,
                          verbose=False, **cfg["fit"])
    reg_models[name] = m

plot_history(reg_hists, "val_loss", "Validation loss on noisy data")
plot_history(reg_hists, "train_loss", "Training loss")

In [ ]:
pd.DataFrame({
    name: {
        "epochs_run": len(reg_hists[name]["val_loss"]),
        "best_val_loss": min(reg_hists[name]["val_loss"]),
        "val_acc_noisy": accuracy(m, X_val_noisy, y_val),
        "test_acc_noisy": accuracy(m, X_test_noisy, y_test),
        "test_acc_clean": accuracy(m, X_test, y_test),
    }
    for name, m in reg_models.items()
}).T

In [ ]:
demo = RegMLP(n_features, n_classes, dropout=0.5, batch_norm=True).to(device)
xb = X_val[:8].to(device)
demo.train()
a1, a2 = demo(xb), demo(xb)
demo.eval()
b1, b2 = demo(xb), demo(xb)
print("train mode, two passes equal:", torch.allclose(a1, a2))
print("eval mode, two passes equal:", torch.allclose(b1, b2))

## 5. Transfer learning with a pretrained CNN

A torchvision ResNet-18 pretrained on ImageNet receives the 12×12 symptom grid, upsampled to 64×64 and replicated to 3 channels. First only the new classification head is trained, then the last residual stage is unfrozen for fine-tuning.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights


class SymptomResNet(nn.Module):
    def __init__(self, n_classes, size=64):
        super().__init__()
        self.backbone = resnet18(weights=ResNet18_Weights.DEFAULT)
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, n_classes)
        self.size = size
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def set_trainable(self, prefixes):
        for name, p in self.backbone.named_parameters():
            p.requires_grad = any(name.startswith(pref) for pref in prefixes)

    def forward(self, x):
        x = x.repeat(1, 3, 1, 1)
        x = F.interpolate(x, size=(self.size, self.size), mode="nearest")
        x = (x - self.mean) / self.std
        return self.backbone(x)


tl_model = SymptomResNet(n_classes)
tl_model.set_trainable(["fc"])
print("trainable (head only):", count_params(tl_model))

In [ ]:
tl_train_loader = make_loader(G_tr, y_tr, batch_size=64, shuffle=True)
tl_val_loader = make_loader(G_val, y_val, batch_size=128)

head_hist = fit(tl_model, tl_train_loader, tl_val_loader, epochs=5, lr=1e-3)
print("test accuracy after head training:", accuracy(tl_model, G_test, y_test))

In [ ]:
tl_model.set_trainable(["fc", "layer4"])
print("trainable (head + layer4):", count_params(tl_model))

ft_hist = fit(tl_model, tl_train_loader, tl_val_loader, epochs=4, lr=1e-4)
print("test accuracy after fine-tuning:", accuracy(tl_model, G_test, y_test))
plot_history({"head only": head_hist, "fine-tune": ft_hist}, "val_acc", "Transfer learning validation accuracy")

In [ ]:
scratch = SymptomResNet(n_classes)
scratch.backbone.apply(
    lambda m: m.reset_parameters() if hasattr(m, "reset_parameters") else None
)
scratch.set_trainable(["fc", "layer1", "layer2", "layer3", "layer4", "conv1", "bn1"])
scratch_hist = fit(scratch, tl_train_loader, tl_val_loader, epochs=5, lr=1e-3)
print("test accuracy from scratch:", accuracy(scratch, G_test, y_test))
plot_history(
    {"pretrained head": head_hist, "pretrained fine-tune": ft_hist, "from scratch": scratch_hist},
    "val_acc", "Pretrained vs scratch"
)

## 6. Hugging Face transformers basics

In [ ]:
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSequenceClassification, pipeline
)

MODEL_NAME = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

example = "Patient reports itching, skin rash, nodal skin eruptions"
enc = tok(example, return_tensors="pt")
print(enc["input_ids"])
print(tok.convert_ids_to_tokens(enc["input_ids"][0]))
print(enc["attention_mask"])

In [ ]:
base = AutoModel.from_pretrained(MODEL_NAME)
with torch.no_grad():
    out = base(**enc)
print(out.last_hidden_state.shape)
print(sum(p.numel() for p in base.parameters()) / 1e6, "M parameters")

In [ ]:
fill = pipeline("fill-mask", model=MODEL_NAME)
for r in fill("The patient has a high [MASK] and a severe headache.")[:3]:
    print(round(r["score"], 3), r["sequence"])

In [ ]:
def to_text(X):
    texts = []
    for row in X:
        idx = torch.nonzero(row).squeeze(1).tolist()
        names = [feature_cols[j].replace("_", " ").strip() for j in idx]
        texts.append("Patient reports " + ", ".join(names))
    return texts


T_tr, T_val, T_test = to_text(X_tr), to_text(X_val), to_text(X_test)
print(T_tr[0])


def encode(texts, max_length=96):
    return tok(texts, padding="max_length", truncation=True,
               max_length=max_length, return_tensors="pt")


e_tr, e_val, e_test = encode(T_tr), encode(T_val), encode(T_test)
print(e_tr["input_ids"].shape)

In [ ]:
hf_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=n_classes,
    id2label={i: c for i, c in enumerate(le.classes_)},
    label2id={c: i for i, c in enumerate(le.classes_)},
).to(device)

hf_train = DataLoader(
    TensorDataset(e_tr["input_ids"], e_tr["attention_mask"], y_tr),
    batch_size=32, shuffle=True,
)


@torch.no_grad()
def hf_accuracy(model, enc_, y, batch_size=64):
    model.eval()
    preds = []
    for i in range(0, len(y), batch_size):
        ids = enc_["input_ids"][i:i + batch_size].to(device)
        mask = enc_["attention_mask"][i:i + batch_size].to(device)
        preds.append(model(input_ids=ids, attention_mask=mask).logits.argmax(1).cpu())
    return (torch.cat(preds) == y).float().mean().item()


optimizer = torch.optim.AdamW(hf_model.parameters(), lr=5e-5, weight_decay=0.01)
EPOCHS = 2
total_steps = EPOCHS * len(hf_train)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lambda s: min(1.0, (s + 1) / (0.1 * total_steps)) * max(0.0, 1 - s / total_steps)
)

for epoch in range(1, EPOCHS + 1):
    hf_model.train()
    running, t0 = 0.0, time.time()
    for ids, mask, yb in hf_train:
        ids, mask, yb = ids.to(device), mask.to(device), yb.to(device)
        loss = hf_model(input_ids=ids, attention_mask=mask, labels=yb).loss
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(hf_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        running += loss.item() * len(yb)
    print(f"epoch {epoch} | loss {running / len(y_tr):.4f} | "
          f"val acc {hf_accuracy(hf_model, e_val, y_val):.3f} | {time.time() - t0:.0f}s")

print("test accuracy:", hf_accuracy(hf_model, e_test, y_test))

In [ ]:
clf = pipeline("text-classification", model=hf_model, tokenizer=tok, top_k=3)
clf("Patient reports continuous sneezing, shivering, chills, watering from eyes")

In [ ]:
hf_model.save_pretrained("symptom-distilbert")
tok.save_pretrained("symptom-distilbert")
reloaded_clf = pipeline("text-classification", model="symptom-distilbert")
reloaded_clf("Patient reports vomiting, yellowish skin, dark urine, abdominal pain")

## 7. Using LLM APIs

Set any of these environment variables before running: `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `HF_TOKEN`. Ollama needs a local server (`ollama serve`) and a pulled model (`ollama pull llama3.2`). Providers that are not configured are skipped. The local Hugging Face model downloads about 1 GB on first use.

In [ ]:
import requests

SYSTEM = (
    "You are a decision-support demo for an educational notebook. "
    "Reply with exactly one disease name taken from the candidate list and nothing else."
)
LABELS = ", ".join(le.classes_)


def build_prompt(symptoms):
    return f"Symptoms: {symptoms}\n\nCandidate diagnoses: {LABELS}\n\nMost likely diagnosis:"


def match_label(text):
    t = text.strip().strip(".").lower()
    for c in le.classes_:
        if c.lower() == t:
            return c
    for c in sorted(le.classes_, key=len, reverse=True):
        if c.lower() in t:
            return c
    return None


def symptoms_of(row):
    return ", ".join(feature_cols[j].replace("_", " ").strip()
                     for j in torch.nonzero(row).squeeze(1).tolist())

In [ ]:
def call_openai(system, prompt, model="gpt-4o-mini"):
    from openai import OpenAI
    client = OpenAI()
    r = client.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=30,
    )
    return r.choices[0].message.content


def call_anthropic(system, prompt, model="claude-sonnet-5-5"):
    import anthropic
    client = anthropic.Anthropic()
    r = client.messages.create(
        model=model,
        max_tokens=30,
        system=system,
        messages=[{"role": "user", "content": prompt}],
    )
    return r.content[0].text


def call_hf_inference(system, prompt, model="meta-llama/Llama-3.1-8B-Instruct"):
    from huggingface_hub import InferenceClient
    client = InferenceClient(token=os.environ["HF_TOKEN"])
    r = client.chat_completion(
        model=model,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}],
        max_tokens=30,
    )
    return r.choices[0].message.content


def call_ollama(system, prompt, model="llama3.2", host="http://localhost:11434"):
    r = requests.post(
        f"{host}/api/chat",
        json={
            "model": model,
            "stream": False,
            "options": {"temperature": 0},
            "messages": [{"role": "system", "content": system}, {"role": "user", "content": prompt}],
        },
        timeout=120,
    )
    r.raise_for_status()
    return r.json()["message"]["content"]


_local_pipe = {}

def call_local_hf(system, prompt, model="Qwen/Qwen2.5-0.5B-Instruct"):
    if model not in _local_pipe:
        _local_pipe[model] = pipeline("text-generation", model=model)
    messages = [{"role": "system", "content": system}, {"role": "user", "content": prompt}]
    out = _local_pipe[model](messages, max_new_tokens=30, do_sample=False)
    return out[0]["generated_text"][-1]["content"]


def ollama_up(host="http://localhost:11434"):
    try:
        return requests.get(f"{host}/api/tags", timeout=1.5).ok
    except Exception:
        return False


providers = {"local_hf": call_local_hf}
if os.environ.get("OPENAI_API_KEY"):
    providers["openai"] = call_openai
if os.environ.get("ANTHROPIC_API_KEY"):
    providers["anthropic"] = call_anthropic
if os.environ.get("HF_TOKEN"):
    providers["hf_inference"] = call_hf_inference
if ollama_up():
    providers["ollama"] = call_ollama
print(list(providers))

In [ ]:
row = X_test[0]
prompt = build_prompt(symptoms_of(row))
print("true label:", le.classes_[y_test[0]])
for name, fn in providers.items():
    try:
        reply = fn(SYSTEM, prompt)
        print(f"{name:14s} -> {reply!r} -> {match_label(reply)}")
    except Exception as e:
        print(f"{name:14s} failed: {type(e).__name__}: {e}")

In [ ]:
N_EVAL = 20
records = []
for name, fn in providers.items():
    correct, answered = 0, 0
    for i in range(N_EVAL):
        try:
            reply = fn(SYSTEM, build_prompt(symptoms_of(X_test[i])))
        except Exception:
            continue
        pred = match_label(reply)
        answered += 1
        correct += int(pred == le.classes_[y_test[i]])
    records.append({"provider": name, "answered": answered,
                    "accuracy": correct / answered if answered else float("nan")})

llm_results = pd.DataFrame(records).set_index("provider")
llm_results.loc["distilbert (fine-tuned)", ["answered", "accuracy"]] = [
    len(y_test), hf_accuracy(hf_model, e_test, y_test)
]
llm_results

### Structured output

In [ ]:
JSON_SYSTEM = (
    "You are a decision-support demo for an educational notebook. "
    "Return only a JSON object with keys: top_3 (list of objects with 'disease' and 'confidence' between 0 and 1), "
    "and 'reasoning' (one sentence). Diseases must come from the candidate list."
)


def extract_json(text):
    start, end = text.find("{"), text.rfind("}")
    return json.loads(text[start:end + 1])


def ask_json(fn, symptoms):
    raw = fn(JSON_SYSTEM, build_prompt(symptoms) + "\nRespond with JSON only.")
    return extract_json(raw)


for name in ("anthropic", "openai", "hf_inference", "ollama"):
    if name in providers:
        fn = providers[name]
        break
else:
    name, fn = "local_hf", providers["local_hf"]

print("using", name)
try:
    print(json.dumps(ask_json(fn, symptoms_of(X_test[5])), indent=2))
except Exception as e:
    print(type(e).__name__, e)

### Streaming and multi-turn

In [ ]:
def stream_anthropic(messages, system, model="claude-sonnet-5-5", max_tokens=300):
    import anthropic
    client = anthropic.Anthropic()
    with client.messages.stream(model=model, max_tokens=max_tokens, system=system, messages=messages) as s:
        for text in s.text_stream:
            print(text, end="", flush=True)
    print()


def stream_openai(messages, system, model="gpt-4o-mini", max_tokens=300):
    from openai import OpenAI
    client = OpenAI()
    stream = client.chat.completions.create(
        model=model, max_tokens=max_tokens, stream=True,
        messages=[{"role": "system", "content": system}] + messages,
    )
    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            print(delta, end="", flush=True)
    print()


def stream_ollama(messages, system, model="llama3.2", host="http://localhost:11434"):
    with requests.post(
        f"{host}/api/chat",
        json={"model": model, "stream": True,
              "messages": [{"role": "system", "content": system}] + messages},
        stream=True, timeout=300,
    ) as r:
        for line in r.iter_lines():
            if line:
                print(json.loads(line).get("message", {}).get("content", ""), end="", flush=True)
    print()


EXPLAIN_SYSTEM = "You explain machine learning results to students in a few short sentences."
history = [{"role": "user", "content": f"A classifier labelled these symptoms as '{le.classes_[y_test[0]]}': {symptoms_of(X_test[0])}. Why might that be plausible?"}]

if "anthropic" in providers:
    stream_anthropic(history, EXPLAIN_SYSTEM)
elif "openai" in providers:
    stream_openai(history, EXPLAIN_SYSTEM)
elif "ollama" in providers:
    stream_ollama(history, EXPLAIN_SYSTEM)
else:
    print("no streaming provider configured")

In [ ]:
class ChatSession:
    def __init__(self, provider, system):
        self.provider = provider
        self.system = system
        self.messages = []

    def say(self, text):
        self.messages.append({"role": "user", "content": text})
        if self.provider == "anthropic":
            import anthropic
            r = anthropic.Anthropic().messages.create(
                model="claude-sonnet-5-5", max_tokens=300,
                system=self.system, messages=self.messages)
            reply = r.content[0].text
        elif self.provider == "openai":
            from openai import OpenAI
            r = OpenAI().chat.completions.create(
                model="gpt-4o-mini", max_tokens=300,
                messages=[{"role": "system", "content": self.system}] + self.messages)
            reply = r.choices[0].message.content
        elif self.provider == "ollama":
            r = requests.post("http://localhost:11434/api/chat", json={
                "model": "llama3.2", "stream": False,
                "messages": [{"role": "system", "content": self.system}] + self.messages}, timeout=300)
            reply = r.json()["message"]["content"]
        else:
            raise ValueError("provider not configured")
        self.messages.append({"role": "assistant", "content": reply})
        return reply


chat_provider = next((p for p in ("anthropic", "openai", "ollama") if p in providers), None)
if chat_provider:
    chat = ChatSession(chat_provider, EXPLAIN_SYSTEM)
    print(chat.say("What is the difference between dropout and batch normalization?"))
    print()
    print(chat.say("Which of the two helps more when the training set is tiny?"))
else:
    print("no chat provider configured")

## Summary

In [ ]:
summary = pd.DataFrame({
    "model": ["MLP", "CNN (12x12 grid)", "RNN", "LSTM", "GRU",
              "ResNet-18 head only", "ResNet-18 fine-tuned", "DistilBERT"],
    "test_acc": [
        accuracy(mlp, X_test, y_test),
        accuracy(cnn, G_test, y_test),
        accuracy(seq_models["rnn"], S_test, y_test),
        accuracy(seq_models["lstm"], S_test, y_test),
        accuracy(seq_models["gru"], S_test, y_test),
        head_hist["val_acc"][-1],
        accuracy(tl_model, G_test, y_test),
        hf_accuracy(hf_model, e_test, y_test),
    ],
})
summary